<a href="https://colab.research.google.com/github/GuardinTheDev/Is-This-Text-Ai-/blob/Model-E%C4%9Fitimi/ModelE%C4%9Fitimi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

In [ ]:
from datasets import load_dataset, DatasetDict

print("Hugging Face Hub'dan 'Yunij/kaggle-comp-daigt' yükleniyor...")
# 'Yunij/kaggle-comp-daigt' veri seti yükleniyor.
raw_dataset = load_dataset('Yunij/kaggle-comp-daigt')

# Veri setinin bölümlerini kontrol edelim ve 'train' ile 'validation' olarak ayarlayalım.
# Eğer doğrudan 'train' ve 'test' bölümleri varsa bunları kullanırız.
# Eğer sadece 'train' varsa, onu böleriz.
if isinstance(raw_dataset, DatasetDict):
    if 'train' in raw_dataset and 'test' in raw_dataset:
        dataset = {
            'train': raw_dataset['train'],
            'validation': raw_dataset['test']
        }
    elif 'train' in raw_dataset:
        # Sadece 'train' bölümü varsa, bunu eğitim ve doğrulama olarak bölelim.
        print("Sadece 'train' bölümü bulundu, eğitim ve doğrulama olarak bölünüyor...")
        split_data = raw_dataset['train'].train_test_split(test_size=0.1, seed=42)
        dataset = {
            'train': split_data['train'],
            'validation': split_data['test']
        }
    else:
        raise ValueError("Veri setinde 'train' bölümü bulunamadı.")
else:
    # Eğer raw_dataset bir DatasetDict değilse ve sadece bir Dataset ise (örneğin sadece train split)
    print("Yüklenen veri seti tek bir split içeriyor, eğitim ve doğrulama olarak bölünüyor...")
    split_data = raw_dataset.train_test_split(test_size=0.1, seed=42)
    dataset = {
        'train': split_data['train'],
        'validation': split_data['test']
    }

print("Yunij/kaggle-comp-daigt Veri Seti Başarıyla Yüklendi ve Bölümleri Ayarlandı!")
print("Eğitim seti boyutu:", len(dataset['train']))
print("Doğrulama seti boyutu:", len(dataset['validation']))
print("\nÖrnek bir veri (eğitim setinden):", dataset['train'][0])
print("\nÖrnek bir veri (doğrulama setinden):", dataset['validation'][0])

In [ ]:
from transformers import AutoTokenizer

# Model altyapısını hazır veri setine bağlama adımı
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_fonksiyonu(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=512)

# Hafızadaki yerel veri setimizin 'train' ve 'validation' bölümlerini tokenlaştırıyoruz
tokenized_datasets = {
    'train': dataset['train'].map(tokenize_fonksiyonu, batched=True),
    'validation': dataset['validation'].map(tokenize_fonksiyonu, batched=True)
}
print("Tokenlaştırma tamamlandı, model eğitimine hazır!")

In [ ]:
import torch
import numpy as np
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

# 1. Cihaz Kontrolü (Colab'da T4 GPU seçiliyse 'cuda' aktif olur, yoksa 'cpu' çalışır)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Eğitim için kullanılan cihaz: {device}")

# 2. Modeli Sınıflandırma İçin Yüklüyoruz
# İnsan (0) ve Yapay Zeka (1) olmak üzere 2 sınıfımız olduğu için num_labels=2 yapıyoruz
model_name = "distilbert-base-uncased"
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)

# 3. Model Başarısını Ölçmek İçin Metrik Fonksiyonu
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='binary')
    return {"accuracy": acc, "f1": f1}

# 4. Eğitim Hiperparametreleri (Ayarları)
training_args = TrainingArguments(
    output_dir="./ai_detector_local_results", # Çıktıların kaydedileceği klasör
    learning_rate=2e-5,                       # Transformer modelleri için ideal öğrenme oranı
    per_device_train_batch_size=16,            # Örnek sayımız çok az olduğu için küçük tuttuk
    per_device_eval_batch_size=16,
    num_train_epochs=3,                       # Modelin veriyi kaç tur döneceği (Epoch)
    weight_decay=0.01,
    eval_strategy="epoch",                    # Her epoch sonunda doğruluğu test et
    save_strategy="epoch",
    load_best_model_at_end=True,              # En başarılı modeli hafızada tut
    logging_steps=1,                          # Logları hemen görmek için 1 yaptık
    report_to="none"                          # Harici raporlama araçlarını kapat
)

# 5. Trainer (Eğitici) Kurulumu
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    compute_metrics=compute_metrics,
)

# 6. Eğitimi Başlatıyoruz
print("\n--- Model Eğitimi Başlıyor ---")
trainer.train()

# 7. Modeli ve Tokenizer'ı Yerel Klasöre Kaydetme
drive_kayit_yolu = "/content/drive/MyDrive/en_iyi_detektor_modeli"

trainer.save_model(drive_kayit_yolu)
tokenizer.save_pretrained(drive_kayit_yolu)
print(f"\nEğitim tamamlandı ve model Google Drive'ınıza ({drive_kayit_yolu}) başarıyla kaydedildi!")
#trainer.save_model("./en_iyi_detektor_modeli")
#tokenizer.save_pretrained("./en_iyi_detektor_modeli")
#print("\nEğitim tamamlandı ve model './en_iyi_detektor_modeli' klasörüne kaydedildi!")

In [ ]:


import torch
import os
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Modelin Google Drive'daki klasör yolu
model_path = "/content/drive/MyDrive/en_iyi_detektor_modeli"

# Eğer Drive bağlı değilse otomatik bağlamaya çalışsın
if not os.path.exists("/content/drive"):
    print("Drive bağlı değil, bağlanılıyor...")
    from google.colab import drive
    drive.mount('/content/drive')

# Klasör kontrolü
if not os.path.exists(model_path):
    print(f"HATA: Google Drive'da '{model_path}' klasörü bulunamadı!")
    print("Lütfen önce modeli eğittiğinizden ve Drive'a kaydettiğinizden emin olun.")
else:
    print("Model Google Drive'dan yükleniyor, lütfen bekleyin...")
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Model ve Tokenizer Yükleme
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)
    model.eval() # Modeli test moduna alıyoruz
    print("Model başarıyla yüklendi! Analize hazır.\n")

    def metni_analiz_et(metin):
        inputs = tokenizer(metin, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        logits = outputs.logits
        olasiliklar = torch.softmax(logits, dim=-1)[0]
        tahmin_id = torch.argmax(logits, dim=-1).item()

        siniflar = {0: "İnsan Tarafından Yazılmış", 1: "Yapay Zeka Tarafından Yazılmış"}

        insan_skoru = olasiliklar[0].item() * 100
        yz_skoru = olasiliklar[1].item() * 100

        return siniflar[tahmin_id], insan_skoru, yz_skoru

    # --- Canlı Test Ekranı ---
    print("=== YAPAY ZEKA METİN DETEKTÖRÜ ===")
    print("Çıkış yapmak için küçük 'q' harfi yazıp Enter'a basın.")

    while True:
        kullanici_metni = input("\nAnaliz edilecek metni girin:\n> ")

        if kullanici_metni.strip().lower() == 'q':
            print("Program kapatıldı.")
            break

        if not kullanici_metni.strip():
            print("Lütfen boş bırakmayın.")
            continue

        karar, insan_yuzde, yz_yuzde = metni_analiz_et(kullanici_metni)

        print("\n" + "="*40)
        print(f"📊 SONUÇ: {karar}")
        print(f"👨‍💻 İnsan Yazısı İhtimali: %{insan_yuzde:.2f}")
        print(f"🤖 Yapay Zeka İhtimali: %{yz_yuzde:.2f}")
        print("="*40)
